# GAT-v4 | Phase 1 — Data Acquisition & Sample Alignment

**Covers plan sections 2.1–2.3, 2.4, 3.1–3.2**

Since all source data is already downloaded, this notebook:
1. Aggregates ~611 GDC STAR-count TSVs → `raw_counts_matrix.csv`
2. Strips Ensembl version suffixes (`.15` etc.)
3. Maps Ensembl IDs → HGNC gene symbols
4. Classifies samples as tumor / normal / other via barcode parsing
5. Loads CNV data from cBioPortal
6. Loads clinical data (OS, age, stage, grade)
7. Builds the master sample alignment table
8. Writes Phase 1 audit YAML

**Expected runtime:** ~15–25 min (dominated by reading 611 TSV files)

**Outputs written to:** `gat-v4/data/processed/`

---
| Cell | What |
|------|------|
| 1 | Imports & path configuration |
| 2 | Load GDC sample sheet |
| 3 | Scan RNA folder — build file map |
| 4 | Preview single TSV — confirm file structure |
| 5 | Aggregate counts matrix (all files, tqdm) |
| 6 | Strip Ensembl version numbers |
| 7 | Download HGNC complete set |
| 8 | Apply HGNC mapping — report rate |
| 9 | Barcode classification (tumor / normal / other) |
| 10 | Load CNV data |
| 11 | Load clinical data (patient + sample) |
| 12 | Build sample alignment table |
| 13 | Save all processed outputs |
| 14 | Phase 1 audit YAML |

---
## Cell 1 — Imports & Path Configuration
Edit only the `PATH CONFIGURATION` block. Never hardcode paths in later cells.

In [17]:
import sys, warnings, hashlib, time, gzip
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import requests
import yaml
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

# ── PATH CONFIGURATION ────────────────────────────────────────────────────────
PROJECT_ROOT  = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")

RNA_MAIN_DIR  = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\gene expression")
RNA_SHEET     = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\gdc_sample_sheet.2026-07-05.tsv")

CNV_DIR       = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\CNV")
CNV_FILE      = CNV_DIR / "data_cna.txt"
CLIN_PATIENT  = CNV_DIR / "data_clinical_patient.txt"
CLIN_SAMPLE   = CNV_DIR / "data_clinical_sample.txt"
# ─────────────────────────────────────────────────────────────────────────────

# Derived output paths (under DC PROJECT/gat-v4/)
KIRC_ROOT    = PROJECT_ROOT / "KIRC"
DATA_PROC     = KIRC_ROOT / "data" / "processed"
DATA_RAW_HGNC = KIRC_ROOT / "data" / "raw" / "hgnc"
AUDIT_DIR     = KIRC_ROOT / "audit"
LOG_DIR       = KIRC_ROOT / "results" / "logs"

for d in [DATA_PROC, DATA_RAW_HGNC, AUDIT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Logger
if str(KIRC_ROOT) not in sys.path:
    sys.path.insert(0, str(KIRC_ROOT))
from src.utils.logger import get_logger
log = get_logger('phase1_data')

# Source data existence check
checks = {
    'RNA main dir'      : RNA_MAIN_DIR,
    'GDC sample sheet'  : RNA_SHEET,
    'CNV data_CNA.txt'  : CNV_FILE,
    'Clinical patient'  : CLIN_PATIENT,
    'Clinical sample'   : CLIN_SAMPLE,
}
all_ok = True
print("Source data checks:")
for label, path in checks.items():
    ok = path.exists()
    print(f"  {'OK' if ok else 'MISSING'}  {label:<25} {path}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("All source paths found. Proceed to Cell 2.")
else:
    print("Fix missing paths above before continuing.")

2026-07-06 16:42:13 | INFO     | phase1_data | Logger initialised. Log file → C:\Users\jiyad\OneDrive\Desktop\PROJECT\gat-v4\results\logs\phase1_data_20260706_164213.log


Source data checks:
  OK  RNA main dir              C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\gene expression
  OK  GDC sample sheet          C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\gdc_sample_sheet.2026-07-05.tsv
  OK  CNV data_CNA.txt          C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\CNV\data_cna.txt
  OK  Clinical patient          C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\CNV\data_clinical_patient.txt
  OK  Clinical sample           C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\CNV\data_clinical_sample.txt

All source paths found. Proceed to Cell 2.


---
## Cell 2 — Load GDC Sample Sheet

Maps **File ID → Sample ID**. Expected ~611 rows.

In [4]:
sample_sheet = pd.read_csv(RNA_SHEET, sep='\t', dtype=str)
sample_sheet.columns = sample_sheet.columns.str.strip()

print(f"Sample sheet shape   : {sample_sheet.shape}")
print(f"Columns              : {list(sample_sheet.columns)}")
print()
display(sample_sheet.head(3))

REQUIRED_COLS = ['File ID', 'File Name', 'Sample ID']
missing_cols  = [c for c in REQUIRED_COLS if c not in sample_sheet.columns]

if missing_cols:
    print(f"Missing columns: {missing_cols}")
    print("Update REQUIRED_COLS to match the column names printed above.")
else:
    print(f"Unique File IDs   : {sample_sheet['File ID'].nunique()}")
    print(f"Unique Sample IDs : {sample_sheet['Sample ID'].nunique()}")
    if 'Sample Type' in sample_sheet.columns:
        print(f"\nSample Type breakdown:")
        print(sample_sheet['Sample Type'].value_counts().to_string())
    log.info(f"Sample sheet loaded: {sample_sheet.shape[0]} rows")
    print("\nCell 2 done.")

Sample sheet shape   : (614, 11)
Columns              : ['File ID', 'File Name', 'Data Category', 'Data Type', 'Project ID', 'Case ID', 'Sample ID', 'Tissue Type', 'Tumor Descriptor', 'Specimen Type', 'Preservation Method']



,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
0,44c858ac-1ce0-491a-aef5-979224aa09f5,144cc9bb-bd29-4986-89a7-5855168dac8e.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-KIRC,TCGA-B8-4146,TCGA-B8-4146-01B,Tumor,Primary,Unknown,Unknown
1,3bca5269-8bb3-4826-bc22-e7b068853f45,44dce722-f50f-4b75-bcf4-aac9afd39e41.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-KIRC,TCGA-BP-4977,TCGA-BP-4977-01A,Tumor,Primary,Unknown,Unknown
2,b65b4ebc-e8b9-4926-8761-56b99fbd9e9a,19f6bda2-39bb-4733-8efc-a4626afb6bd1.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-KIRC,TCGA-BP-4981,TCGA-BP-4981-01A,Tumor,Primary,Unknown,Unknown


2026-07-06 14:52:11 | INFO     | phase1_data | Sample sheet loaded: 614 rows


Unique File IDs   : 614
Unique Sample IDs : 610

Cell 2 done.


---
## Cell 3 — Scan RNA Folder & Build File Map

Expected structure: `kirc_rawdata_rna/<file_uuid>/<filename>.tsv`  
Falls back to glob if a file is not in its expected subfolder.

In [5]:
file_id_to_path = {}
not_found = []

for _, row in sample_sheet.iterrows():
    fid  = row['File ID']
    fname= row['File Name']

    # Primary: uuid subfolder (standard GDC layout)
    candidate = RNA_MAIN_DIR / fid / fname
    if candidate.exists():
        file_id_to_path[fid] = candidate
        continue

    # Fallback 1: flat layout
    flat = RNA_MAIN_DIR / fname
    if flat.exists():
        file_id_to_path[fid] = flat
        continue

    # Fallback 2: deep glob (slower but handles any nesting)
    matches = list(RNA_MAIN_DIR.rglob(fname))
    if matches:
        file_id_to_path[fid] = matches[0]
    else:
        not_found.append((fid, fname))

print(f"File IDs in sheet       : {len(sample_sheet)}")
print(f"Files located on disk   : {len(file_id_to_path)}")
print(f"Files NOT found         : {len(not_found)}")

if not_found:
    print(f"\nMissing (first 5):")
    for fid, fname in not_found[:5]:
        print(f"  {fid}  /  {fname}")

sample_map = sample_sheet[sample_sheet['File ID'].isin(file_id_to_path)].copy()
sample_map['file_path'] = sample_map['File ID'].map(file_id_to_path)

miss_frac = len(not_found) / len(sample_sheet)
print()
if miss_frac == 0:
    print("All files found.")
elif miss_frac < 0.02:
    print(f"{len(not_found)} files missing (<2%). Proceeding with found files.")
    log.warning(f"{len(not_found)} files not found")
else:
    print(f"{len(not_found)} files missing ({miss_frac:.0%}). Check download completeness.")

log.info(f"File map: {len(file_id_to_path)} files located")
print("\nCell 3 done.")

2026-07-06 14:52:22 | INFO     | phase1_data | File map: 582 files located


File IDs in sheet       : 614
Files located on disk   : 582
Files NOT found         : 32

Missing (first 5):
  e1beebe9-dc24-4f9d-a89f-87001fcf25c7  /  b93e9aab-9d4c-4e1e-9a04-b2b858b32fa8.rna_seq.augmented_star_gene_counts.tsv
  c69d9c36-61e3-4985-b17b-b64bd7ed3821  /  eb8069ea-756b-4d4c-9c14-dd2b8b2a7af6.rna_seq.augmented_star_gene_counts.tsv
  01851b83-f8c4-4108-99c9-248fe29e39e9  /  6dd12311-581d-46fd-9b98-6523b67ec2e4.rna_seq.augmented_star_gene_counts.tsv
  986759bd-7232-46a9-84f3-0e71026a01fd  /  05791785-80ef-46cd-970b-5bd0c0d1b8e3.rna_seq.augmented_star_gene_counts.tsv
  e6a09aba-4d6e-4079-9b34-0e9caf705885  /  05468136-c936-455f-b5dd-75062f8df0fd.rna_seq.augmented_star_gene_counts.tsv

32 files missing (5%). Check download completeness.

Cell 3 done.


---
## Cell 4 — Preview Single TSV: Confirm File Structure

Read the raw lines AND the parsed DataFrame.  
Confirm `HEADER_ROW` before running Cell 5.

Expected:
- Row 0: metadata / comment line  
- Row 1 (`HEADER_ROW=1`): column names  
- Rows 2–5: summary rows (N_unmapped, N_multimapping, N_noFeature, N_ambiguous)  
- Rows 6+: ENSG gene rows

In [6]:
# ── Adjust if structure differs from expected ─────────────────────────────────
HEADER_ROW = 1    # 0-indexed row that contains column names
# ─────────────────────────────────────────────────────────────────────────────

example_row  = sample_map.iloc[0]
example_fid  = example_row['File ID']
example_path = file_id_to_path[example_fid]
example_sid  = example_row['Sample ID']
is_gzip      = str(example_path).endswith('.gz')

print(f"File : {example_path.name}  ({example_path.stat().st_size/1024:.0f} KB)")
print(f"SID  : {example_sid}")
print()

# Raw lines
open_fn = gzip.open if is_gzip else open
print("=== RAW LINES (first 12) ===")
with open_fn(example_path, 'rt', encoding='utf-8', errors='replace') as fh:
    for i, line in enumerate(fh):
        if i >= 12:
            break
        print(f"  [{i:02d}]  {line.rstrip()[:130]}")

# Parsed
print(f"\n=== PARSED with header=row {HEADER_ROW} ===")
df_preview = pd.read_csv(
    example_path, sep='\t', header=HEADER_ROW, dtype=str,
    compression='gzip' if is_gzip else 'infer'
)
GENE_COL = df_preview.columns[0]
print(f"Total rows : {len(df_preview):,}  |  Columns : {list(df_preview.columns)}")
print()
display(df_preview.head(8))

ensg_count    = df_preview[GENE_COL].str.startswith('ENSG', na=False).sum()
summary_count = df_preview[GENE_COL].str.startswith('N_',   na=False).sum()
print(f"\nENSG rows   : {ensg_count:,}   (expected ~58,000–60,000)")
print(f"N_* rows    : {summary_count}    (expected 4)")

if ensg_count > 50000:
    print(f"\nStructure confirmed. HEADER_ROW={HEADER_ROW} is correct.")
else:
    print(f"\nFewer ENSG rows than expected — adjust HEADER_ROW (current={HEADER_ROW}).")

File : 144cc9bb-bd29-4986-89a7-5855168dac8e.rna_seq.augmented_star_gene_counts.tsv  (4130 KB)
SID  : TCGA-B8-4146-01B

=== RAW LINES (first 12) ===
  [00]  # gene-model: GENCODE v36
  [01]  gene_id	gene_name	gene_type	unstranded	stranded_first	stranded_second	tpm_unstranded	fpkm_unstranded	fpkm_uq_unstranded
  [02]  N_unmapped			1322671	1322671	1322671
  [03]  N_multimapping			3181251	3181251	3181251
  [04]  N_noFeature			3700530	30569284	30601371
  [05]  N_ambiguous			4198728	924654	918451
  [06]  ENSG00000000003.15	TSPAN6	protein_coding	5898	2920	2978	101.1091	25.6034	25.7172
  [07]  ENSG00000000005.6	TNMD	protein_coding	38	18	20	2.0020	0.5069	0.5092
  [08]  ENSG00000000419.13	DPM1	protein_coding	3222	1625	1597	207.5758	52.5635	52.7972
  [09]  ENSG00000000457.14	SCYL3	protein_coding	910	790	778	10.2807	2.6033	2.6149
  [10]  ENSG00000000460.17	C1orf112	protein_coding	358	501	567	4.6630	1.1808	1.1860
  [11]  ENSG00000000938.13	FGR	protein_coding	478	236	243	10.9904	2.7830	2.7954

=== P

,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,N_unmapped,NaN,NaN,1322671,1322671,1322671,NaN,NaN,NaN
1,N_multimapping,NaN,NaN,3181251,3181251,3181251,NaN,NaN,NaN
2,N_noFeature,NaN,NaN,3700530,30569284,30601371,NaN,NaN,NaN
3,N_ambiguous,NaN,NaN,4198728,924654,918451,NaN,NaN,NaN
4,ENSG00000000003.15,TSPAN6,protein_coding,5898,2920,2978,101.1091,25.6034,25.7172
5,ENSG00000000005.6,TNMD,protein_coding,38,18,20,2.0020,0.5069,0.5092
6,ENSG00000000419.13,DPM1,protein_coding,3222,1625,1597,207.5758,52.5635,52.7972
7,ENSG00000000457.14,SCYL3,protein_coding,910,790,778,10.2807,2.6033,2.6149



ENSG rows   : 60,660   (expected ~58,000–60,000)
N_* rows    : 4    (expected 4)

Structure confirmed. HEADER_ROW=1 is correct.


---
## Cell 5 — Aggregate Counts Matrix

Reads all files. Keeps only ENSG rows. Extracts `unstranded` count column.  
Assembles **(gene_id × sample_id)** matrix.

**Runtime: ~15–20 minutes.** Failed files are logged and skipped.

In [7]:
# Update COUNT_COL if your files use a different column name (check Cell 4 output)
COUNT_COL = 'unstranded'

print(f"Gene column  : '{GENE_COL}'")
print(f"Count column : '{COUNT_COL}'")
print(f"Files        : {len(sample_map)}")
print()

if COUNT_COL not in df_preview.columns:
    raise ValueError(
        f"'{COUNT_COL}' not in TSV columns {list(df_preview.columns)}. "
        "Update COUNT_COL above."
    )

count_series_list = []
failed_files      = []
t0 = time.time()

for _, row in tqdm(sample_map.iterrows(), total=len(sample_map), desc='Reading TSVs'):
    fid   = row['File ID']
    sid   = row['Sample ID']
    fpath = file_id_to_path[fid]
    is_gz = str(fpath).endswith('.gz')

    try:
        df = pd.read_csv(
            fpath, sep='\t', header=HEADER_ROW,
            usecols=[GENE_COL, COUNT_COL],
            dtype={GENE_COL: str, COUNT_COL: float},
            compression='gzip' if is_gz else 'infer'
        )
        # Drop summary rows — keep only gene rows
        df = df[df[GENE_COL].str.startswith('ENSG', na=False)]
        count_series_list.append(df.set_index(GENE_COL)[COUNT_COL].rename(sid))

    except Exception as e:
        failed_files.append((fid, sid, str(e)))
        log.warning(f"SKIP {sid}: {e}")

elapsed = time.time() - t0
print(f"\nRead {len(count_series_list)} files in {elapsed/60:.1f} min")
if failed_files:
    print(f"Failed/skipped: {len(failed_files)}")
    for _, sid, err in failed_files[:3]:
        print(f"  {sid}: {err}")

print("Assembling matrix...")
counts_matrix = pd.concat(count_series_list, axis=1)
print(f"Matrix shape : {counts_matrix.shape}   (expected ~60,000 x ~611)")

nan_pct = counts_matrix.isna().sum().sum() / counts_matrix.size * 100
print(f"NaN fraction : {nan_pct:.3f}%  (expected ~0%)")
if nan_pct > 1.0:
    print("  High NaN — some samples may have different gene sets. Investigate.")

log.info(f"Counts matrix assembled: {counts_matrix.shape}")
print("\nCell 5 done.")

Gene column  : 'gene_id'
Count column : 'unstranded'
Files        : 582



Reading TSVs: 100%|██████████| 582/582 [01:20<00:00,  7.26it/s]



Read 582 files in 1.3 min
Assembling matrix...


2026-07-06 14:53:49 | INFO     | phase1_data | Counts matrix assembled: (60660, 582)


Matrix shape : (60660, 582)   (expected ~60,000 x ~611)
NaN fraction : 0.000%  (expected ~0%)

Cell 5 done.


---
## Cell 6 — Strip Ensembl Version Numbers

`ENSG00000000003.15` → `ENSG00000000003`

Collisions (two versioned IDs → same base ID) are resolved by summing.

In [8]:
print("Examples before stripping:")
for v in [x for x in counts_matrix.index[:10] if '.' in x][:5]:
    print(f"  {v}  ->  {v.split('.')[0]}")

counts_matrix.index = counts_matrix.index.str.split('.').str[0]

n_dup = counts_matrix.index.duplicated().sum()
print(f"\nGenes before dedup : {len(counts_matrix):,}")
print(f"Duplicate base IDs : {n_dup}")

if n_dup > 0:
    print(f"  Resolving by summing (PAR_Y duplicates, readthrough overlaps).")
    counts_matrix = counts_matrix.groupby(counts_matrix.index).sum()

# Drop any residual non-ENSG rows
non_ensg = counts_matrix.index[~counts_matrix.index.str.startswith('ENSG')]
if len(non_ensg):
    counts_matrix = counts_matrix[counts_matrix.index.str.startswith('ENSG')]
    print(f"  Dropped {len(non_ensg)} non-ENSG rows.")

print(f"Final shape : {counts_matrix.shape}")
print(f"Index sample: {list(counts_matrix.index[:5])}")
print("\nCell 6 done.")

Examples before stripping:
  ENSG00000000003.15  ->  ENSG00000000003
  ENSG00000000005.6  ->  ENSG00000000005
  ENSG00000000419.13  ->  ENSG00000000419
  ENSG00000000457.14  ->  ENSG00000000457
  ENSG00000000460.17  ->  ENSG00000000460

Genes before dedup : 60,660
Duplicate base IDs : 44
  Resolving by summing (PAR_Y duplicates, readthrough overlaps).
Final shape : (60616, 582)
Index sample: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460']

Cell 6 done.


---
## Cell 7 — Download HGNC Complete Set

Downloads the HGNC approved gene table and builds Ensembl → symbol mapping.
Cached locally after first download.

**Runtime: ~30 sec first run, instant after.**

In [9]:
HGNC_URL  = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
HGNC_FILE = DATA_RAW_HGNC / "hgnc_complete_set.txt"

if HGNC_FILE.exists():
    print(f"Using cached HGNC file: {HGNC_FILE}")
    print(f"Size   : {HGNC_FILE.stat().st_size/1024**2:.1f} MB")
    print(f"Date   : {datetime.fromtimestamp(HGNC_FILE.stat().st_mtime).strftime('%Y-%m-%d')}")
    print("(Delete file and re-run to refresh.)")
else:
    print(f"Downloading: {HGNC_URL}")
    r = requests.get(HGNC_URL, timeout=120)
    r.raise_for_status()
    HGNC_FILE.write_bytes(r.content)
    print(f"Downloaded: {HGNC_FILE.stat().st_size/1024**2:.1f} MB")

# Load
hgnc_df = pd.read_csv(
    HGNC_FILE, sep='\t',
    usecols=['symbol', 'ensembl_gene_id', 'status', 'locus_type'],
    dtype=str
)
print(f"\nHGNC rows : {len(hgnc_df):,}")
print(f"Status values : {hgnc_df['status'].dropna().unique().tolist()}")

# Keep Approved entries with valid Ensembl ID
hgnc_ok = hgnc_df[
    (hgnc_df['status'] == 'Approved') &
    hgnc_df['ensembl_gene_id'].notna() &
    hgnc_df['ensembl_gene_id'].str.startswith('ENSG', na=False)
].copy()

print(f"Approved with Ensembl ID : {len(hgnc_ok):,}")

# Resolve 1-Ensembl → many-symbols (keep alphabetically first)
n_dup_ensg = hgnc_ok['ensembl_gene_id'].duplicated().sum()
if n_dup_ensg:
    print(f"Duplicate Ensembl IDs: {n_dup_ensg} → keeping first by symbol sort")
    hgnc_ok = hgnc_ok.sort_values('symbol').drop_duplicates('ensembl_gene_id', keep='first')

ensembl_to_symbol = dict(zip(hgnc_ok['ensembl_gene_id'], hgnc_ok['symbol']))
print(f"Mapping dict : {len(ensembl_to_symbol):,} entries")
print("Examples:")
for eid, sym in list(ensembl_to_symbol.items())[:5]:
    print(f"  {eid}  ->  {sym}")

log.info(f"HGNC mapping built: {len(ensembl_to_symbol)} entries")
print("\nCell 7 done.")

Downloading: https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt
Downloaded: 16.1 MB

HGNC rows : 45,003
Status values : ['Approved']
Approved with Ensembl ID : 42,321
Duplicate Ensembl IDs: 3 → keeping first by symbol sort
Mapping dict : 42,318 entries
Examples:
  ENSG00000121410  ->  A1BG
  ENSG00000268895  ->  A1BG-AS1
  ENSG00000148584  ->  A1CF
  ENSG00000175899  ->  A2M
  ENSG00000245105  ->  A2M-AS1


2026-07-06 14:53:57 | INFO     | phase1_data | HGNC mapping built: 42318 entries



Cell 7 done.


---
## Cell 8 — Apply HGNC Mapping

Expected mapping rate: **85–92%**.  
Unmapped IDs (pseudogenes, deprecated loci, etc.) are dropped.

In [10]:
n_before       = len(counts_matrix)
mapped_symbols = counts_matrix.index.map(ensembl_to_symbol)

n_mapped   = mapped_symbols.notna().sum()
n_unmapped = mapped_symbols.isna().sum()
map_rate   = n_mapped / n_before * 100

print(f"Genes in matrix       : {n_before:,}")
print(f"Mapped to HGNC symbol : {n_mapped:,}  ({map_rate:.1f}%)")
print(f"Unmapped (dropped)    : {n_unmapped:,}  ({100-map_rate:.1f}%)")

if map_rate < 80:
    print()
    print("Mapping rate below 80% — possible causes:")
    print("  1. Ensembl version mismatch (GDC data uses older Ensembl)")
    print("  2. HGNC file outdated — delete and re-run Cell 7")

unmapped_examples = counts_matrix.index[mapped_symbols.isna()].tolist()
print(f"\nUnmapped examples: {unmapped_examples[:8]}")

# Apply mapping
counts_mapped = counts_matrix[mapped_symbols.notna()].copy()
counts_mapped.index = mapped_symbols[mapped_symbols.notna()].values
counts_mapped.index.name = 'gene_symbol'

# Resolve symbol collisions (two Ensembl IDs → same symbol)
n_sym_dup = counts_mapped.index.duplicated().sum()
if n_sym_dup:
    print(f"\nDuplicate symbols: {n_sym_dup} — resolving by summing.")
    counts_mapped = counts_mapped.groupby(counts_mapped.index).sum()

print(f"\nFinal mapped matrix : {counts_mapped.shape}")
print(f"Gene examples       : {list(counts_mapped.index[:10])}")

# Save mapping table for audit
mapping_used = hgnc_ok[
    hgnc_ok['ensembl_gene_id'].isin(counts_matrix.index[mapped_symbols.notna()])
][['ensembl_gene_id', 'symbol', 'locus_type']].copy()
mapping_used.to_csv(DATA_PROC / 'ensembl_to_symbol_map.csv', index=False)
print("Mapping table -> data/processed/ensembl_to_symbol_map.csv")

log.info(f"HGNC mapping applied: {n_before} -> {len(counts_mapped)} genes ({map_rate:.1f}% mapped)")
print("\nCell 8 done.")

Genes in matrix       : 60,616
Mapped to HGNC symbol : 41,924  (69.2%)
Unmapped (dropped)    : 18,692  (30.8%)

Mapping rate below 80% — possible causes:
  1. Ensembl version mismatch (GDC data uses older Ensembl)
  2. HGNC file outdated — delete and re-run Cell 7

Unmapped examples: ['ENSG00000002079', 'ENSG00000093100', 'ENSG00000100101', 'ENSG00000103200', 'ENSG00000106540', 'ENSG00000108516', 'ENSG00000111780', 'ENSG00000112096']


2026-07-06 14:53:58 | INFO     | phase1_data | HGNC mapping applied: 60616 -> 41924 genes (69.2% mapped)



Final mapped matrix : (41924, 582)
Gene examples       : ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'FIRRM', 'FGR', 'CFH', 'FUCA2', 'GCLC', 'NFYA']
Mapping table -> data/processed/ensembl_to_symbol_map.csv

Cell 8 done.


---
## Cell 9 — Barcode Classification

**Critical (v3 bug fix):** Classification uses the barcode 4th field, not the GDC label.

Format: `TCGA-XX-YYYY-ZZA` where `ZZ` is:
- `01`–`09` → tumor
- `10`–`19` → normal

Expected: ~535 tumor, ~72 normal.

In [18]:
def classify_sample(sample_id: str) -> str:
    try:
        code = sample_id.split('-')[3][:2]
        n = int(code)
        if 1 <= n <= 9:   return 'tumor'
        if 10 <= n <= 19: return 'normal'
    except Exception:
        pass
    return 'other'

def patient_from_sample(sample_id: str) -> str:
    parts = sample_id.split('-')
    return '-'.join(parts[:3]) if len(parts) >= 3 else sample_id

sample_map = sample_map.copy()
sample_map['tissue_type'] = sample_map['Sample ID'].apply(classify_sample)
sample_map['patient_id']  = sample_map['Sample ID'].apply(patient_from_sample)

print("Tissue type (barcode-parsed):")
print(sample_map['tissue_type'].value_counts().to_string())

# Cross-check against GDC label if available
if 'Sample Type' in sample_map.columns:
    print("\nGDC Sample Type (reference only, not used for classification):")
    print(sample_map['Sample Type'].value_counts().to_string())
    gdc_tumor     = sample_map['Sample Type'].str.lower().str.contains('tumor', na=False)
    barcode_tumor = sample_map['tissue_type'] == 'tumor'
    disagree      = (gdc_tumor != barcode_tumor).sum()
    if disagree:
        print(f"\n{disagree} samples disagree — barcode classification takes precedence.")
    else:
        print("\nGDC labels and barcode parsing agree.")

n_tumor  = (sample_map['tissue_type'] == 'tumor').sum()
n_normal = (sample_map['tissue_type'] == 'normal').sum()
n_other  = (sample_map['tissue_type'] == 'other').sum()
print(f"\nTumor  : {n_tumor}   (expected ~535)")
print(f"Normal : {n_normal}   (expected ~72)")
print(f"Other  : {n_other}   (will be excluded)")

# Drop 'other'
sample_map = sample_map[sample_map['tissue_type'] != 'other'].copy()

log.info(f"Barcode classification: {n_tumor} tumor, {n_normal} normal, {n_other} excluded")
print("\nCell 9 done.")

2026-07-06 16:42:56 | INFO     | phase1_data | Barcode classification: 513 tumor, 69 normal, 0 excluded


Tissue type (barcode-parsed):
tissue_type
tumor     513
normal     69

Tumor  : 513   (expected ~535)
Normal : 69   (expected ~72)
Other  : 0   (will be excluded)

Cell 9 done.


---
## Cell 10 — Load CNV Data

cBioPortal `data_CNA.txt` — GISTIC2 thresholded integers (-2, -1, 0, 1, 2).  
Sample columns are short barcodes (`TCGA-XX-YYYY-ZZ`).

In [19]:
cnv_raw = pd.read_csv(CNV_FILE, sep='\t', dtype=str, low_memory=False)
cnv_raw.columns = cnv_raw.columns.str.strip()

print(f"CNV raw shape : {cnv_raw.shape}")
print(f"First cols    : {list(cnv_raw.columns[:6])}")
print(f"Last cols     : {list(cnv_raw.columns[-3:])}")
print()

GENE_COL_CNV = 'Hugo_Symbol'
if GENE_COL_CNV not in cnv_raw.columns:
    print(f"'{GENE_COL_CNV}' not found. Columns: {list(cnv_raw.columns[:10])}")
    print("Update GENE_COL_CNV.")
else:
    cnv = cnv_raw.set_index(GENE_COL_CNV).drop(columns=['Entrez_Gene_Id'], errors='ignore')
    cnv = cnv.apply(pd.to_numeric, errors='coerce')

    print(f"CNV matrix shape : {cnv.shape}")
    vals = sorted(cnv.stack().dropna().astype(int).unique().tolist())
    print(f"GISTIC2 values   : {vals}   (expected [-2, -1, 0, 1, 2])")

    print(f"\nSample ID examples (CNV) : {list(cnv.columns[:4])}")
    print("  Format: TCGA-XX-YYYY-ZZ  (shorter than full RNA barcode — normal)")

    cnv_patients = set('-'.join(col.split('-')[:3]) for col in cnv.columns)
    print(f"Unique patients in CNV : {len(cnv_patients)}")

    log.info(f"CNV loaded: {cnv.shape}")
    print("\nCell 10 done.")

CNV raw shape : (24776, 530)
First cols    : ['Hugo_Symbol', 'Entrez_Gene_Id', 'TCGA-3Z-A93Z-01', 'TCGA-6D-AA2E-01', 'TCGA-A3-3306-01', 'TCGA-A3-3307-01']
Last cols     : ['TCGA-MM-A84U-01', 'TCGA-MW-A4EC-01', 'TCGA-T7-A92I-01']

CNV matrix shape : (24776, 528)


2026-07-06 16:43:13 | INFO     | phase1_data | CNV loaded: (24776, 528)


GISTIC2 values   : [-2, -1, 0, 1, 2]   (expected [-2, -1, 0, 1, 2])

Sample ID examples (CNV) : ['TCGA-3Z-A93Z-01', 'TCGA-6D-AA2E-01', 'TCGA-A3-3306-01', 'TCGA-A3-3307-01']
  Format: TCGA-XX-YYYY-ZZ  (shorter than full RNA barcode — normal)
Unique patients in CNV : 528

Cell 10 done.


---
## Cell 11 — Load Clinical Data

cBioPortal files have `#`-prefixed comment/metadata rows — use `comment='#'` to skip cleanly.

Target fields: `PATIENT_ID`, `OS_MONTHS`, `OS_STATUS`, `AGE`, `AJCC_PATHOLOGIC_TUMOR_STAGE`, `GRADE`

In [20]:
# ── Load both clinical files ──────────────────────────────────────────────────
clin_pt   = pd.read_csv(CLIN_PATIENT, sep='\t', comment='#', dtype=str, low_memory=False)
clin_samp = pd.read_csv(CLIN_SAMPLE,  sep='\t', comment='#', dtype=str, low_memory=False)
clin_pt.columns   = clin_pt.columns.str.strip().str.upper()
clin_samp.columns = clin_samp.columns.str.strip().str.upper()

print(f"Clinical patient : {clin_pt.shape}   cols: {list(clin_pt.columns)}")
print(f"Clinical sample  : {clin_samp.shape}  cols: {list(clin_samp.columns)}")
print()

# ── Find column names (handles cBioPortal version variation) ──────────────────
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns: return c
    return None

COL_PAT   = find_col(clin_pt, ['PATIENT_ID', '#PATIENT_ID', 'PATIENT ID'])
COL_OS_MO = find_col(clin_pt, ['OS_MONTHS', 'OS_STATUS_MONTHS', 'OVERALL_SURVIVAL'])
COL_OS_ST = find_col(clin_pt, ['OS_STATUS', 'VITAL_STATUS'])
COL_AGE   = find_col(clin_pt, ['AGE', 'DIAGNOSIS_AGE', 'AGE_AT_DIAGNOSIS'])
COL_STAGE = find_col(clin_pt, ['AJCC_PATHOLOGIC_TUMOR_STAGE', 'TUMOR_STAGE', 'STAGE'])
COL_GRADE = find_col(clin_pt, ['GRADE', 'TUMOR_GRADE', 'NEOPLASM_HISTOLOGIC_GRADE'])

print("Detected column mappings:")
for name, col in [('PATIENT_ID', COL_PAT), ('OS_MONTHS', COL_OS_MO),
                  ('OS_STATUS', COL_OS_ST), ('AGE', COL_AGE),
                  ('STAGE', COL_STAGE), ('GRADE', COL_GRADE)]:
    status = 'OK ' if col else 'MISSING'
    print(f"  {status}  {name:<30} -> {col}")

missing_req = [n for n, c in [('PATIENT_ID', COL_PAT), ('OS_MONTHS', COL_OS_MO),
                                ('OS_STATUS', COL_OS_ST)] if c is None]
if missing_req:
    print(f"\nRequired columns missing: {missing_req}")
    print("Update find_col() candidates above.")
else:
    clin_pt[COL_PAT] = clin_pt[COL_PAT].str.lstrip('#').str.strip()

    def encode_event(val):
        if pd.isna(val): return np.nan
        v = str(val).upper()
        if '1' in v or 'DECEASED' in v or 'DEAD' in v: return 1
        if '0' in v or 'LIVING'   in v or 'ALIVE' in v: return 0
        return np.nan

    def encode_stage(val):
        if pd.isna(val):
            return np.nan

        v = str(val).upper().strip()

        if v.startswith("STAGE IV"):
            return 4
        elif v.startswith("STAGE III"):
            return 3
        elif v.startswith("STAGE II"):
            return 2
        elif v.startswith("STAGE I"):
            return 1

        return np.nan

    def encode_grade(val):
        if pd.isna(val): return np.nan
        v = str(val).upper().strip()
        for i in [1, 2, 3, 4]:
            if f'G{i}' in v or f'GRADE {i}' in v or str(i) in v: return i
        return np.nan

    clin_pt['event']    = clin_pt[COL_OS_ST].apply(encode_event)
    clin_pt['os_months']= pd.to_numeric(clin_pt[COL_OS_MO], errors='coerce')
    if COL_AGE:   clin_pt['age']   = pd.to_numeric(clin_pt[COL_AGE],   errors='coerce')
    if COL_STAGE: clin_pt['stage'] = clin_pt[COL_STAGE].apply(encode_stage)
    if COL_GRADE: clin_pt['grade'] = clin_pt[COL_GRADE].apply(encode_grade)

    required_present = ['event', 'os_months'] + (['stage'] if COL_STAGE else [])
    keep_cols = [COL_PAT, 'event', 'os_months'] + \
                (['age'] if COL_AGE else []) + \
                (['stage'] if COL_STAGE else []) + \
                (['grade'] if COL_GRADE else [])

    clin_clean = clin_pt[
        clin_pt[required_present].notna().all(axis=1) &
        (clin_pt['os_months'] > 0)
    ][keep_cols].copy().rename(columns={COL_PAT: 'patient_id'})
    clin_clean = clin_clean.drop_duplicates('patient_id')

    print(f"\nPatients total    : {len(clin_pt)}")
    print(f"Complete records  : {len(clin_clean)}   (expected ~520)")
    print(f"Events (deceased) : {int(clin_clean['event'].sum())}")
    print(f"Censored (alive)  : {int((clin_clean['event']==0).sum())}")

    if COL_STAGE:
        print(f"\nStage distribution:")
        print(clin_clean['stage'].value_counts().sort_index().to_string())
    if COL_GRADE:
        print(f"\nGrade distribution:")
        print(clin_clean['grade'].value_counts().sort_index().to_string())

    log.info(f"Clinical: {len(clin_clean)} patients with complete records")
    print("\nCell 11 done.")

2026-07-06 16:43:26 | INFO     | phase1_data | Clinical: 532 patients with complete records


Clinical patient : (537, 60)   cols: ['OTHER_PATIENT_ID', 'PATIENT_ID', 'FORM_COMPLETION_DATE', 'HISTOLOGICAL_DIAGNOSIS', 'GRADE', 'LATERALITY', 'PROSPECTIVE_COLLECTION', 'RETROSPECTIVE_COLLECTION', 'SEX', 'RACE', 'ETHNICITY', 'HISTORY_OTHER_MALIGNANCY', 'HISTORY_NEOADJUVANT_TRTYN', 'INITIAL_PATHOLOGIC_DX_YEAR', 'LYMPH_NODES_EXAMINED', 'LYMPH_NODE_EXAMINED_COUNT', 'LYMPH_NODES_EXAMINED_POSITIVE', 'AJCC_STAGING_EDITION', 'AJCC_TUMOR_PATHOLOGIC_PT', 'AJCC_NODES_PATHOLOGIC_PN', 'AJCC_METASTASIS_CLINICAL_CM', 'AJCC_METASTASIS_PATHOLOGIC_PM', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'TUMOR_STATUS', 'IDH_LEVEL', 'SERUM_CALCIUM_LEVEL', 'HEMOGLOBIN_LEVEL', 'PLATELET_COUNT', 'WBC', 'ERYTHROCYTE_SEDIMENTATION_RATE', 'TOBACCO_SMOKING_HISTORY_INDICATOR', 'SMOKING_YEAR_STARTED', 'SMOKING_YEAR_STOPPED', 'SMOKING_PACK_YEARS', 'KARNOFSKY_PERFORMANCE_SCORE', 'ECOG_SCORE', 'PERFORMANCE_STATUS_TIMING', 'RADIATION_TREATMENT_ADJUVANT', 'TREATMENT_OUTCOME_FIRST_COURSE', 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'AGE

---
## Cell 12 — Build Sample Alignment Table

Master `sample_alignment.csv` — the single source of truth for inclusion flags.

**Tumor inclusion:** has_rna AND has_cnv AND has_survival  
**Normal inclusion:** has_rna only

In [21]:
rna_samples   = set(counts_mapped.columns)
survival_pats = set(clin_clean['patient_id'])
cnv_patients  = set('-'.join(c.split('-')[:3]) for c in cnv.columns)

align_rows = []
for _, row in sample_map.iterrows():
    sid   = row['Sample ID']
    pid   = row['patient_id']
    ttype = row['tissue_type']

    has_rna      = sid in rna_samples
    has_cnv      = pid in cnv_patients
    has_survival = pid in survival_pats

    include = (
        (has_rna and has_cnv and has_survival) if ttype == 'tumor'
        else has_rna  # normal
    )

    align_rows.append({
        'sample_id'   : sid,
        'patient_id'  : pid,
        'tissue_type' : ttype,
        'has_rna'     : has_rna,
        'has_cnv'     : has_cnv,
        'has_survival': has_survival,
        'include'     : include,
    })

align = pd.DataFrame(align_rows)

tumor_incl  = align[(align['tissue_type']=='tumor')  & align['include']]
normal_incl = align[(align['tissue_type']=='normal') & align['include']]
both_pats   = set(tumor_incl['patient_id']) & set(normal_incl['patient_id'])

print("Alignment breakdown:")
print(align.groupby(['tissue_type', 'include']).size().to_string())
print()
print(f"Included tumor    : {len(tumor_incl)}   (expected ~520)")
print(f"Included normal   : {len(normal_incl)}   (expected ~72)")
print(f"Total included    : {align['include'].sum()}")
print(f"Patients with both: {len(both_pats)}  (used in DESeq2 paired design)")

excl = align[(align['tissue_type']=='tumor') & ~align['include']]
if len(excl):
    print(f"\nExcluded tumor ({len(excl)} samples):")
    print(f"  No CNV      : {(~excl['has_cnv']).sum()}")
    print(f"  No survival : {(~excl['has_survival']).sum()}")
    print(f"  No RNA      : {(~excl['has_rna']).sum()}")

log.info(f"Alignment: {len(tumor_incl)} tumor + {len(normal_incl)} normal included")
print("\nCell 12 done.")

2026-07-06 16:43:47 | INFO     | phase1_data | Alignment: 500 tumor + 69 normal included


Alignment breakdown:
tissue_type  include
normal       True        69
tumor        False       13
             True       500

Included tumor    : 500   (expected ~520)
Included normal   : 69   (expected ~72)
Total included    : 569
Patients with both: 67  (used in DESeq2 paired design)

Excluded tumor (13 samples):
  No CNV      : 8
  No survival : 5
  No RNA      : 0

Cell 12 done.


---
## Cell 13 — Save All Processed Outputs

All files consumed by downstream phases are written here.

**Runtime: ~2–5 min** (counts matrix is large).

In [22]:
outputs = {}

# 1. Counts matrix — included samples only
included_sids = align[align['include']]['sample_id'].tolist()
rna_incl_sids = [s for s in included_sids if s in counts_mapped.columns]
counts_out    = counts_mapped[rna_incl_sids].copy()

print(f"Saving raw_counts_matrix.csv   {counts_out.shape} ...", end='', flush=True)
t0 = time.time()
p  = DATA_PROC / 'raw_counts_matrix.csv'
counts_out.to_csv(p)
print(f" {time.time()-t0:.0f}s  ({p.stat().st_size/1024**2:.0f} MB)")
outputs['raw_counts_matrix'] = p

# 2. Sample alignment
print(f"Saving sample_alignment.csv    {align.shape} ...", end='', flush=True)
t0 = time.time()
p  = DATA_PROC / 'sample_alignment.csv'
align.to_csv(p, index=False)
print(f" {time.time()-t0:.0f}s")
outputs['sample_alignment'] = p

# 3. Clinical
print(f"Saving clinical_clean.csv      {clin_clean.shape} ...", end='', flush=True)
t0 = time.time()
p  = DATA_PROC / 'clinical_clean.csv'
clin_clean.to_csv(p, index=False)
print(f" {time.time()-t0:.0f}s")
outputs['clinical_clean'] = p

# 4. CNV — included tumor patients only
tumor_pats   = set(tumor_incl['patient_id'])
cnv_col_keep = [c for c in cnv.columns if '-'.join(c.split('-')[:3]) in tumor_pats]
cnv_out      = cnv[cnv_col_keep].copy()

print(f"Saving cnv_raw.csv             {cnv_out.shape} ...", end='', flush=True)
t0 = time.time()
p  = DATA_PROC / 'cnv_raw.csv'
cnv_out.to_csv(p)
print(f" {time.time()-t0:.0f}s")
outputs['cnv_raw'] = p

print()
print("Files written:")
for name, path in outputs.items():
    print(f"  {name:<30} {path.stat().st_size/1024**2:.1f} MB")

log.info(f"All Phase 1 outputs saved to {DATA_PROC}")
print("\nCell 13 done.")

Saving raw_counts_matrix.csv   (41924, 577) ... 18s  (116 MB)
Saving sample_alignment.csv    (582, 7) ... 0s
Saving clinical_clean.csv      (532, 6) ... 0s
Saving cnv_raw.csv             (24776, 491) ...

2026-07-06 16:44:35 | INFO     | phase1_data | All Phase 1 outputs saved to C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\processed


 3s

Files written:
  raw_counts_matrix              115.6 MB
  sample_alignment               0.0 MB
  clinical_clean                 0.0 MB
  cnv_raw                        24.9 MB

Cell 13 done.


---
## Cell 14 — Phase 1 Audit YAML

Saved to `audit/phase1_data_acquisition.yaml`. Commit this file.  
A reviewer asking "which version of the data?" is answered here.

In [23]:
def file_md5(path, chunk=8192):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()

print("Computing checksums...", end='', flush=True)
checksum_files = ['sample_alignment.csv', 'clinical_clean.csv', 'cnv_raw.csv']
checksums = {fname: file_md5(DATA_PROC / fname) for fname in checksum_files}
print(" done")

audit = {
    'phase'     : 'Phase 1 — Data Acquisition & Sample Alignment',
    'timestamp' : datetime.now().isoformat(),
    'rna_data': {
        'source'          : str(RNA_MAIN_DIR),
        'sample_sheet'    : str(RNA_SHEET),
        'files_in_sheet'  : int(len(sample_sheet)),
        'files_found'     : int(len(file_id_to_path)),
        'files_failed'    : int(len(failed_files)),
        'header_row'      : HEADER_ROW,
        'count_column'    : COUNT_COL,
    },
    'gene_mapping': {
        'method'           : 'HGNC complete set (Approved, ensembl_gene_id)',
        'source_url'       : HGNC_URL,
        'local_file'       : str(HGNC_FILE),
        'download_date'    : datetime.fromtimestamp(HGNC_FILE.stat().st_mtime).strftime('%Y-%m-%d'),
        'genes_before'     : int(n_before),
        'genes_after'      : int(len(counts_mapped)),
        'mapping_rate_pct' : round(map_rate, 2),
        'version_stripping': 'ENSGXXX.N -> ENSGXXX',
    },
    'sample_classification': {
        'method'         : 'TCGA barcode 4th field; overrides GDC label',
        'tumor_included' : int(len(tumor_incl)),
        'normal_included': int(len(normal_incl)),
        'excluded_other' : int(n_other),
        'paired_patients': int(len(both_pats)),
    },
    'cnv_data': {
        'source'          : str(CNV_FILE),
        'method'          : 'GISTIC2.0 thresholded (cBioPortal)',
        'genes'           : int(cnv_out.shape[0]),
        'tumor_samples'   : int(cnv_out.shape[1]),
        'normal_cnv'      : 'diploid assumption (CNV=0)',
    },
    'clinical_data': {
        'source_patient'  : str(CLIN_PATIENT),
        'source_sample'   : str(CLIN_SAMPLE),
        'patients_total'  : int(len(clin_pt)),
        'patients_complete': int(len(clin_clean)),
        'events_deceased' : int(clin_clean['event'].sum()),
        'censored'        : int((clin_clean['event']==0).sum()),
        'stage_encoding'  : 'Roman I-IV -> integer 1-4',
        'grade_encoding'  : 'G1-G4 -> integer 1-4',
        'os_encoding'     : '1:DECEASED->1, 0:LIVING->0',
    },
    'output_checksums': checksums,
    'next_phase' : 'Phase 3 — DESeq2 (04_deseq2.R)',
    'status'     : 'COMPLETE',
}

audit_path = AUDIT_DIR / 'phase1_data_acquisition.yaml'
with open(audit_path, 'w', encoding='utf-8') as f:
    yaml.dump(audit, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"Audit YAML -> {audit_path}")
print()
print(yaml.dump(audit, default_flow_style=False, sort_keys=False))

print("=" * 55)
print("PHASE 1 COMPLETE")
print("=" * 55)
print(f"  Counts matrix : {counts_out.shape[0]:,} genes x {counts_out.shape[1]} samples")
print(f"  HGNC mapped   : {len(counts_mapped):,} genes ({map_rate:.1f}%)")
print(f"  Tumor         : {len(tumor_incl)} samples")
print(f"  Normal        : {len(normal_incl)} samples")
print(f"  Clinical      : {len(clin_clean)} patients")
print(f"  CNV           : {cnv_out.shape}")
print()
print("Next: 04_deseq2.R — DESeq2 differential expression + VST normalisation")

Computing checksums... done
Audit YAML -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\audit\phase1_data_acquisition.yaml

phase: "Phase 1 \u2014 Data Acquisition & Sample Alignment"
timestamp: '2026-07-06T16:44:41.534915'
rna_data:
  source: C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\gene expression
  sample_sheet: C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\gdc_sample_sheet.2026-07-05.tsv
  files_in_sheet: 614
  files_found: 582
  files_failed: 0
  header_row: 1
  count_column: unstranded
gene_mapping:
  method: HGNC complete set (Approved, ensembl_gene_id)
  source_url: https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt
  local_file: C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\raw\hgnc\hgnc_complete_set.txt
  download_date: '2026-07-06'
  genes_before: 60616
  genes_after: 41924
  mapping_rate_pct: !!python/object/apply:numpy.core.multiarray.scalar
  - !!python/object/apply:numpy.dtype
    args:
    - f8
    - false
    - true
    